# Tutorial on Genetic Epidemiology

## Introduction

In this workshop, we will start by converting a `*.vcf` file to PLINK2 `*.psam`, `*.pvar`, and `*.pgen` files. We will then explore genetic ancestry using principal component analysis and visualize results from a genome-wide association study.

We will use a **small 1000 Genomes teaching dataset distributed by Hail** (data science tool for cloud computing), but we will perform the genetics analyses with **PLINK2** and use **Python** for data handling and visualization. 

> ## How to use this notebook
>
> This notebook contains **two kinds of executable cells**:
>
> - **Python cells** run directly with the notebook's Python kernel.
> - **Terminal / Bash cells** begin with `%%bash`. Everything in that cell is executed by the shell, so commands such as `plink2`, `ls`, `awk`, `grep`, and `wget` work there.
>
> You **can mix terminal and Python work in the same notebook**, but it is best to keep them in **separate cells**. A cell beginning with `%%bash` is entirely Bash; a normal code cell is Python. For a single shell command inside a Python cell, Jupyter also supports `!command`, but this workshop keeps the two environments separate for clarity.
>
> Make sure the notebook, helper functions, and data files are in the expected working directory, and that `plink2` is available on your shell `PATH` before running the analysis.

This is roughly the tutorial workflow:  

```text
Explore 1000 Genomes genotypes
        |
        v
Sample + variant QC
        |
        v
LD pruning + PCA
        |
        v
Ancestry visualization
        |
        v
GWAS of CaffeineConsumption
        |
        v
Evaluate population stratification
        |
        v
Adjusted GWAS
        |
        v
Regional association plot

```

By the end of the workshop, you should be able to:

1. Inspect genotype and phenotype files.
2. Check whether genotype and phenotype sample IDs overlap.
3. Perform basic genotype and sample quality control.
4. Perform LD pruning and principal component analysis (PCA).
5. Visualize genetic ancestry using principal components.
6. Run a GWAS with and without ancestry adjustment.
7. Make Manhattan and Q-Q plots in Python.
8. Examine the strongest association region.

---

# 1. Make sure you have access to the right files

Please, ensure that you have access to the following files and to plink2 binary file (all of them available on GitHub). 

```text
1kg.vcf.bgz
1kg_annotations.txt
ensembl_gene_annotations.txt
```

We will start from a small VCF file, downsampled from 1000 Genomes Project. 

The annotation file contains:

- `Sample`: sample ID
- `Population`: 1000 Genomes population label
- `SuperPopulation`: broad ancestry group
- `isFemale`: sex indicator
- `PurpleHair`: simulated binary phenotype
- `CaffeineConsumption`: simulated quantitative phenotype

## Terminal / Bash
For reference the original files were downloaded using the following commands:

In [ ]:
%%bash
wget -c https://storage.googleapis.com/hail-tutorial/1kg.vcf.bgz
wget -c https://storage.googleapis.com/hail-tutorial/1kg_annotations.txt
wget -c https://storage.googleapis.com/hail-tutorial/ensembl_gene_annotations.txt

Check that these files are present:

In [ ]:
%%bash
ls -lh

---

# 2. Convert the VCF to PLINK2 format (*.psam, *.pvar, *.pgen)

In this workshop we will use data from 1000 Genomes Project, using GRCh37 reference genome, limited to **autosomal chromosomes (chromosomes 1-22)**. This keeps the focus on PCA, GWAS, and PGS and avoids introducing sex-chromosome/PAR handling.

## Terminal / Bash

In [ ]:
%%bash
./plink2 \
  --vcf 1kg.vcf.bgz \
  --set-all-var-ids '@:#:$r:$a' \
  --chr 1-22 \
  --make-pgen \
  --out 1kg

In [ ]:
%%bash
## For mac:

./plink2_mac \
  --vcf 1kg.vcf.bgz \
  --set-all-var-ids '@:#:$r:$a' \
  --chr 1-22 \
  --make-pgen \
  --out 1kg

You should now have:

```text
1kg.pgen
1kg.pvar
1kg.psam
1kg.log
```

Check that PLINK2 created the files:

In [ ]:
%%bash
ls -lh 1kg.pgen 1kg.pvar 1kg.psam

Compare the size to the original *.vcf file

In [ ]:
%%bash
ls -lh 1kg.vcf.bgz

Q1. What is the main advantage of using plink2 file format instead of *.vcf?


---

# 3. Python packages we will use

The Python sections of this notebook use:

- `pandas` for tabular data
- `numpy` for numerical operations
- `matplotlib` for general plotting
- `scipy` for statistical calculations
- `helper functions` for Manhattan and Q-Q plots

---

# 4. Getting to know the genotype data

## Terminal / Bash
Check the PLINK2 version:

In [ ]:
%%bash
./plink2 --version

In [ ]:
%%bash
## For mac:

./plink2_mac --version

Inspect the sample file (*.psam):

In [ ]:
%%bash
head 1kg.psam

Count samples:

In [ ]:
%%bash
awk 'NR>1 {n++} END {print "Samples in 1kg.psam:", n}' 1kg.psam

Inspect the first variants:

In [ ]:
%%bash
head 1kg.pvar

Count variants:

In [ ]:
%%bash
awk '$0 !~ /^#/ {n++} END {print "Autosomal variants in 1kg.pvar:", n}' 1kg.pvar

Q2. How many samples and genetic variants are there?


Count variants by chromosome:

In [ ]:
%%bash
awk '$0 !~ /^#/ {count[$1]++} END {for (c in count) print c, count[c]}' 1kg.pvar | sort -V

---

# 5. Inspect the phenotype and ancestry data

Let's first start by inspecting the phenotype and ancestry data we have from the individuals included.  

## Terminal / Bash

In [ ]:
%%bash
head 1kg_annotations.txt

Count rows:

In [ ]:
%%bash
awk 'NR>1 {n++} END {print "Samples in annotation table:", n}' 1kg_annotations.txt

## Python

In [ ]:
import pandas as pd

ann = pd.read_csv("1kg_annotations.txt", sep="\t")

print("Shape:", ann.shape)
print()
print(ann.head())
print()
print(ann.dtypes)

The broad ancestry labels are stored in `SuperPopulation`:

In [ ]:
print(ann["SuperPopulation"].value_counts())

The five 1000 Genomes super-populations are:

- `AFR`: African
- `AMR`: Admixed American
- `EAS`: East Asian
- `EUR`: European
- `SAS`: South Asian

Inspect the quantitative phenotype:

In [ ]:
print(ann["CaffeineConsumption"].describe())

And the simulated binary phenotype:

In [ ]:
print(ann["PurpleHair"].value_counts())

---

# 6. Do the genotype and phenotype samples overlap?

Never assume that files contain the same individuals simply because they come from the same project. Match samples explicitly by ID.

## Python

In [ ]:
import pandas as pd

psam = pd.read_csv("1kg.psam", sep=r"\s+", dtype=str)
ann = pd.read_csv("1kg_annotations.txt", sep="\t", dtype={"Sample": str})

iid_col = "#IID" if "#IID" in psam.columns else "IID"

geno_ids = set(psam[iid_col])
annotation_ids = set(ann["Sample"])
overlap = geno_ids & annotation_ids

print("Genotype samples:", len(geno_ids))
print("Annotation samples:", len(annotation_ids))
print("Overlap:", len(overlap))
print("Genotype samples without annotations:", len(geno_ids - annotation_ids))
print("Annotation samples outside genotype subset:", len(annotation_ids - geno_ids))

Check duplicates as well:

In [ ]:
print("Duplicate genotype IDs:", psam[iid_col].duplicated().sum())
print("Duplicate annotation IDs:", ann["Sample"].duplicated().sum())

### Question

Q3. Why is it fine for the annotation table to contain many samples that are not present in the genotype subset, while missing annotations for genotype samples would be a problem?

---

# 7. Create the analysis phenotype table

Restrict the 3,500-row Hail annotation table to the 284 genotype samples and preserve the genotype sample order.

## Python

In [ ]:
import pandas as pd

psam = pd.read_csv("1kg.psam", sep=r"\s+", dtype=str)
ann = pd.read_csv("1kg_annotations.txt", sep="\t")

iid_col = "#IID" if "#IID" in psam.columns else "IID"

sample_order = psam[[iid_col]].rename(columns={iid_col: "#IID"})

analysis = sample_order.merge(
    ann,
    left_on="#IID",
    right_on="Sample",
    how="left",
    validate="one_to_one"
)

assert analysis["Sample"].notna().all(), "Some genotype samples are missing annotations"

# Hail stores sex as a Boolean. For a regression covariate we use female=1, male=0.
if analysis["isFemale"].dtype == bool:
    analysis["isFemale_num"] = analysis["isFemale"].astype(int)
else:
    analysis["isFemale_num"] = (
        analysis["isFemale"].astype(str).str.lower()
        .map({"true": 1, "false": 0})
    )

analysis.to_csv("1kg_annotations.txt", sep="\t", index=False)

analysis[["#IID", "CaffeineConsumption"]].to_csv(
    "caffeine.pheno",
    sep="\t",
    index=False
)

analysis[["#IID", "isFemale_num"]].rename(
    columns={"isFemale_num": "isFemale"}
).to_csv(
    "base.covar",
    sep="\t",
    index=False
)

print("Analysis samples:", len(analysis))
print()
print(
    analysis[["#IID_x", "Population", "SuperPopulation", "isFemale", "CaffeineConsumption"]]
    .head()
)

Check the ancestry composition of the actual genotype subset:

In [ ]:
print(analysis["SuperPopulation"].value_counts())

---

# 8. Basic quality control

Quality control is not a single universal recipe. Different datasets require different decisions, and many of these tend to be empirical or based on experience. Here we will calculate several common metrics and apply a small number of simple filters.

## Terminal / Bash
Calculate allele frequencies, missingness, Hardy-Weinberg statistics, and heterozygosity:

- `Allele frequencies`: count the number of (ALT) alleles among the total number of alleles (two per human individual)
- `Missingness`: sample and variant missingness allows us to understand whether some samples or variants had low quality. One report for each of sample/ variant missingnes
- `Hardy-Weinberg statistics`: writes autosomal Hardy-Weinberg equilibrium exact test statistics  
- `Heterozygosity`: computes observed and expected homozygous/heterozygous genotype counts for each sample  

Hary-Weinberg equilibrium describes the genotype frequencies expected in a population when alleles combine randomly, in the absence of evolutionary forces that systematically change those frequencies.  

p= f(A), q= f(a), p + q= 1  

Under HWE, the expected genotype frequencies are (AA + Aa + aa = 1):  

p<sup>2</sup> + 2pq + q<sup>2</sup> = 1  

Suppose that  

**p=0.7** and **q=0.3**  

If alleles are paired randomly, then:  

AA = 0.7 <sup> 2 </sup> = 0.49   
Aa = 2 x (0.7) x (0.3) = 0.42  
aa = 0.3<sup>2</sup> = 0.09

Heterozygosity is computed as `1 - (observed het. count / expected het. count)`: 
- F statistic > 0: indicates excess homozygosity  
- F statistics = 0: indicates heterozygosity is similar to what's expected  
- F statistic < 0: indicates excess heterozygosity  

Excess homozygosity can arise due to inbreeding (consanguinity), different ancestry compared to rest of samples or technical issues. Excess heterozygosity may indicate sample contamination or technical/genotyping artifacts. The expected het count is obtained from Hardy-Weinberg equilibrium accross all SNPs.

We use `plink2` for this, which will output in total 5 reports with the prefix `qc_before`

In [ ]:
%%bash
plink2 \
  --pfile 1kg \
  --freq \
  --missing \
  --hardy \
  --het \
  --out qc_before

In [ ]:
%%bash
## for mac:
./plink2_mac \
  --pfile 1kg \
  --freq \
  --missing \
  --hardy \
  --het \
  --out qc_before

This produces files including:

```text
qc_before.afreq
qc_before.smiss
qc_before.vmiss
qc_before.hardy
qc_before.het
```

Inspect them:

In [ ]:
%%bash
head qc_before.afreq
head qc_before.smiss
head qc_before.vmiss

## Python: sample missingness

Let's inspect sample missingness first. It is well-accepted a threshold of 0.98 for sample missingess, meaning that, for each sample, we will only allow a maximum of 2% of variants to be missing. Cut-offs tend to be general consensus rather than carefully selected, and may vary from group to group, and based on the specific genotyping array used. Here we will filter out genetic samples with missingness >3%.  

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

smiss = pd.read_csv("qc_before.smiss", sep=r"\s+")

print(smiss.head())
print(smiss["F_MISS"].describe())

plt.hist(smiss["F_MISS"], bins=25)
plt.xlabel("Fraction of missing genotypes")
plt.ylabel("Number of samples")
plt.title("Sample missingness")
plt.show()

Q. How many individuals have a genotype missingness > 3%?

## Python: variant missingness (call rate)

Let's inspect sample missingness first. It is well-accepted a threshold of 0.98 for sample missingess, meaning that, for each sample, we will only allow a maximum of 2% of variants to be missing. Cut-offs tend to be general consensus rather than carefully selected, and may vary from group to group, and based on the specific genotyping array used. Here we will filter out genetic samples with missingness >3%.  

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

smiss = pd.read_csv("qc_before.smiss", sep=r"\s+")

print(smiss.head())
print(smiss["F_MISS"].describe())

plt.hist(smiss["F_MISS"], bins=25)
plt.xlabel("Fraction of missing genotypes")
plt.ylabel("Number of samples")
plt.title("Sample missingness")
plt.show()

Q. How many individuals have a genotype missingness > 3%?

## Python: allele-frequency distribution

Genetic variants with low allele frequencies tend to be excluded due to a reduction in power, which further depends on sample size and the effect sizes. Large meta-analyses with millions of samples are starting to go down the allele frequency spectrum, with some including all variants with a minor allele frequency > 0.001%. For this workshop, we will only keep variants with an allele frequency >1%.  

In [ ]:
freq = pd.read_csv("qc_before.afreq", sep=r"\s+")

# ALT_FREQS is numeric for the biallelic SNPs in this dataset.
maf = freq["ALT_FREQS"].astype(float)
maf = maf.where(maf <= 0.5, 1 - maf)

plt.hist(maf, bins=30)
plt.xlabel("Minor allele frequency")
plt.ylabel("Number of variants")
plt.title("Minor allele-frequency spectrum")
plt.show()

### Apply simple filters

For this exercise we will require:

- sample call rate >= 97% (`--mind 0.03`)
- variant call rate >= 97% (`--geno 0.03`)
- minor allele frequency >= 1% (`--maf 0.01`)

## Terminal / Bash

In [ ]:
%%bash
./plink2 \
  --pfile 1kg \
  --mind 0.03 \
  --geno 0.03 \
  --maf 0.01 \
  --make-pgen \
  --out 1kg_qc

In [ ]:
%%bash
#for mac
./plink2_mac \
  --pfile 1kg \
  --mind 0.03 \
  --geno 0.03 \
  --maf 0.01 \
  --make-pgen \
  --out 1kg_qc

Inspect the log:

In [ ]:
%%bash
grep -E "samples|variants|remaining|removed" 1kg_qc.log

We do **not** apply a global Hardy-Weinberg filter here. This dataset contains several ancestries, and population mixture itself can produce departures from Hardy-Weinberg equilibrium. In a real analysis, HWE filtering should be considered in an ancestry-aware manner and in the context of study design.

---

# 9. Linkage disequilibrium and PCA

Nearby variants are correlated because of linkage disequilibrium (LD). PCA is usually calculated from a set of approximately independent common variants so that a few high-LD genomic regions do not dominate the principal components.

PLINK2 explicitly recommends removing very-low-frequency variants and LD-pruning before PCA.

## Terminal / Bash: LD pruning

These commands produce a pruned subset of variants that are in approximate linkage equilibrium with each other. Here we take chunks of 500 kb, and remove variants that have an R<sub>2</sub> >0.2.

In [ ]:
%%bash
./plink2 \
  --pfile 1kg_qc \
  --indep-pairwise 500kb 0.2 \
  --out 1kg_prune

In [ ]:
%%bash
./plink2_mac \
  --pfile 1kg_qc \
  --indep-pairwise 500kb 0.2 \
  --out 1kg_prune

Q. How many variants remain after pruning?

In [ ]:
%%bash
wc -l 1kg_prune.prune.in

## Terminal / Bash: PCA

Calculate ten PCs using the LD-pruned variants:

In [ ]:
%%bash
./plink2 \
  --pfile 1kg_qc \
  --extract 1kg_prune.prune.in \
  --pca 10 \
  --out 1kg_pca

In [ ]:
%%bash
./plink2_mac \
  --pfile 1kg_qc \
  --extract 1kg_prune.prune.in \
  --pca 10 \
  --out 1kg_pca

The main outputs are:

```text
1kg_pca.eigenvec
1kg_pca.eigenval
```

Inspect the first few PC values:

In [ ]:
%%bash
head 1kg_pca.eigenvec

---

# 10. Plot genetic ancestry

We will now combine the principal components with the 1000 Genomes ancestry labels.

## Python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pcs = pd.read_csv("1kg_pca.eigenvec", sep=r"\s+")
ann = pd.read_csv("1kg_annotations.txt", sep="\t")

pc_iid = "#IID" if "#IID" in pcs.columns else "IID"

plot_df = pcs.merge(
    ann[["Sample", "Population", "SuperPopulation"]],
    left_on=pc_iid,
    right_on="Sample",
    how="left"
)

for pop, group in plot_df.groupby("SuperPopulation"):
    plt.scatter(group["PC1"], group["PC2"], label=pop, alpha=0.75)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Genetic principal components")
plt.legend(title="Super-population")
plt.show()

### Questions

1. Do individuals cluster by self-reported super-population?
2. Which ancestry groups appear closest to one another?
3. Which axis defines which ancestry groups?
4. Do you see individuals between major clusters?
5. Why might genetic ancestry be continuous rather than categorical?

Try PC2 versus PC3:

In [ ]:
for pop, group in plot_df.groupby("SuperPopulation"):
    plt.scatter(group["PC2"], group["PC3"], label=pop, alpha=0.75)

plt.xlabel("PC2")
plt.ylabel("PC3")
plt.legend(title="Super-population")
plt.show()

---

# 11. A first GWAS

We will first perform an association test **without covariates**. This is intentional: the purpose is to observe the effect of population stratification.

For a quantitative phenotype, PLINK2 `--glm` fits one linear regression per variant.

## Terminal / Bash

In [ ]:
%%bash
./plink2 \
  --pfile 1kg_qc \
  --pheno caffeine.pheno \
  --pheno-name CaffeineConsumption \
  --glm hide-covar allow-no-covars \
  --out gwas_naive

In [ ]:
%%bash
./plink2_mac \
  --pfile 1kg_qc \
  --pheno caffeine.pheno \
  --pheno-name CaffeineConsumption \
  --glm hide-covar allow-no-covars \
  --out gwas_naive

Find the output file:

In [ ]:
%%bash
ls gwas_naive*.glm.linear

Inspect the strongest associations:

In [ ]:
%%bash
sort -g -k15 gwas_naive.CaffeineConsumption.glm.linear | head

## Python

In [ ]:
import pandas as pd
from glob import glob

naive_file = glob("gwas_naive.*.glm.linear")[0]
naive = pd.read_csv(naive_file, sep=r"\s+")

naive = naive[(naive["TEST"] == "ADD") & naive["P"].notna()].copy()

print(
    naive.sort_values("P")[
        ["#CHROM", "POS", "ID", "A1", "BETA", "SE", "P"]
    ].head(10)
)

---

# 12. Manhattan and Q-Q plots

A Manhattan plot displays association p-value across the genome. A Q-Q plot compares the observed distribution of association P-values with the distribution expected under the null hypothesis (uniform distribution).

## Python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from plot_helpers import manhattan

gwas = pd.read_csv(
    "gwas_naive.CaffeineConsumption.glm.linear",
    sep=r"\s+"
)

gwas = gwas[gwas["TEST"] == "ADD"]

manhattan(
    gwas,
    annotate_top=5
)
plt.show()

### Calculate genomic inflation

The genomic inflation factor, lambda GC, summarizes whether the association test statistics are systematically larger than expected.

## Python

In [ ]:
import numpy as np
from scipy.stats import chi2
from plot_helpers import qqplot
p = gwas["P"].astype(float).clip(lower=1e-300, upper=1)
chisq = chi2.isf(p, df=1)

lambda_gc = np.median(chisq) / chi2.ppf(0.5, df=1)
print("Lambda GC:", round(lambda_gc, 3))

gwas = gwas[gwas["TEST"] == "ADD"]

qqplot(gwas["P"]);
plt.show()

### Questions

1. Does the Q-Q plot follow the diagonal?
2. Are association statistics inflated?
3. Is there a single clear locus, or are many regions apparently associated?
4. What could explain this pattern?

---

# 13. Is the phenotype related to ancestry?

As you probably know by now, the GWAS of `CaffeineConsumption` is largely affected by population stratification. Let's inspect CaffeinConsumption by ancestry label.   

## Python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

analysis = pd.read_csv("1kg_annotations.txt", sep="\t")

order = sorted(analysis["SuperPopulation"].dropna().unique())
values = [
    analysis.loc[analysis["SuperPopulation"] == pop, "CaffeineConsumption"].values
    for pop in order
]

plt.boxplot(values, tick_labels=order)
plt.xlabel("Super-population")
plt.ylabel("Caffeine consumption")
plt.title("Simulated phenotype by ancestry")
plt.show()

### Question

If both allele frequencies and the phenotype differ among populations, what might happen in a GWAS that does not adjust for ancestry?

---

# 14. Add principal components as covariates

PLINK2 writes PC scores in a format that can be used as covariates. We will add sex and the first three PCs to a single covariate file.

## Python

In [ ]:
import pandas as pd

pcs = pd.read_csv("1kg_pca.eigenvec", sep=r"\s+")
analysis = pd.read_csv("1kg_annotations.txt", sep="\t")

pc_iid = "#IID" if "#IID" in pcs.columns else "IID"

covar = pcs.merge(
    analysis[["#IID", "isFemale_num"]],
    left_on=pc_iid,
    right_on="#IID",
    how="left"
)

covar = covar.rename(columns={"isFemale_num": "isFemale"})

covar[["#IID", "isFemale", "PC1", "PC2", "PC3"]].to_csv(
    "gwas.covar",
    sep="\t",
    index=False
)

print(covar[["#IID", "isFemale", "PC1", "PC2", "PC3"]].head())

---

# 15. GWAS adjusted for ancestry

Now repeat the association analysis while adjusting for sex and the first three principal components.

## Terminal / Bash

In [ ]:
%%bash
./plink2 \
  --pfile 1kg_qc \
  --pheno caffeine.pheno \
  --pheno-name CaffeineConsumption \
  --covar gwas.covar \
  --covar-name isFemale,PC1,PC2,PC3 \
  --glm hide-covar \
  --out gwas_adjusted

In [ ]:
%%bash
./plink2_mac \
  --pfile 1kg_qc \
  --pheno caffeine.pheno \
  --pheno-name CaffeineConsumption \
  --covar gwas.covar \
  --covar-name isFemale,PC1,PC2,PC3 \
  --glm hide-covar \
  --out gwas_adjusted

## Python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from plot_helpers import manhattan, qqplot
from glob import glob

adjusted = pd.read_csv(
    "gwas_adjusted.CaffeineConsumption.glm.linear",
    sep=r"\s+"
)

adjusted = adjusted[adjusted["TEST"] == "ADD"]

qqplot(adjusted["P"])
plt.show()

Calculate lambda GC again:

In [ ]:
import numpy as np
from scipy.stats import chi2

p = adjusted["P"].astype(float).clip(lower=1e-300, upper=1)
chisq = chi2.isf(p, df=1)
lambda_gc = np.median(chisq) / chi2.ppf(0.5, df=1)

print("Adjusted lambda GC:", round(lambda_gc, 3))

Inspect the strongest results:

In [ ]:
print(
    adjusted.sort_values("P")[
        ["#CHROM", "POS", "ID", "REF", "ALT", "A1", "BETA", "SE", "P"]
    ].head(15)
)

### Questions

1. How did the Q-Q plot change after including PCs?
2. How did lambda GC change?
3. What happened to the Manhattan plot?
4. Does a localized association signal remain?
5. Why can including ancestry PCs remove false-positive associations?

---

# 16. Compare naïve and adjusted P-values

## Python

In [ ]:
comparison = gwas[["ID", "P"]].rename(columns={"P": "P_naive"}).merge(
    adjusted[["ID", "P"]].rename(columns={"P": "P_adjusted"}),
    on="ID"
)

import numpy as np
import matplotlib.pyplot as plt

plt.scatter(
    -np.log10(comparison["P_naive"]),
    -np.log10(comparison["P_adjusted"]),
    alpha=0.5
)

lim = max(
    -np.log10(comparison["P_naive"]).max(),
    -np.log10(comparison["P_adjusted"]).max()
)
plt.plot([0, lim], [0, lim], linestyle="--")
plt.xlabel("Naive -log10(P)")
plt.ylabel("Adjusted -log10(P)")
plt.title("Effect of population-structure adjustment")
plt.show()

Points far below the diagonal were much more significant before ancestry adjustment than afterward.

---

# 17. Zoom in on the strongest association region

## Python

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lead = adjusted.loc[adjusted["P"].idxmin()]

chrom = str(lead["#CHROM"])
pos = int(lead["POS"])
window = 1_000_000

region = adjusted[
    (adjusted["#CHROM"].astype(str) == chrom) &
    (adjusted["POS"] >= pos - window) &
    (adjusted["POS"] <= pos + window)
].copy()

print("Lead variant")
print(lead[["#CHROM", "POS", "ID", "A1", "BETA", "P"]])

plt.scatter(region["POS"] / 1e6, -np.log10(region["P"]), alpha=0.7)
plt.scatter(
    [pos / 1e6],
    [-np.log10(float(lead["P"]))],
    s=100,
    marker="*",
    label=str(lead["ID"])
)
plt.xlabel(f"Chromosome {chrom} position (Mb)")
plt.ylabel("-log10(P)")
plt.title("Regional association plot")
plt.legend()
plt.show()

### Optional: find genes in this region

If you downloaded `ensembl_gene_annotations.txt`, inspect its columns:

In [ ]:
genes = pd.read_csv("ensembl_gene_annotations.txt", sep="\t")
print(genes.columns.tolist())
print(genes.head())

The Ensembl gene annotation file contains chromosome, gene start, gene end, and gene name fields. Find genes overlapping the region around the lead SNP:

In [ ]:
genes["Chromosome"] = genes["Chromosome"].astype(str)

nearby_genes = genes[
    (genes["Chromosome"] == chrom) &
    (genes["Gene end"] >= pos - window) &
    (genes["Gene start"] <= pos + window)
].copy()

print(
    nearby_genes[["Gene name", "Gene start", "Gene end"]]
    .sort_values("Gene start")
    .to_string(index=False)
)

> Association does not prove that the closest gene is causal. Fine-mapping and functional follow-up are separate problems.

---

# Optional extension: binary phenotype

The phenotype file also contains a simulated binary phenotype, `PurpleHair`.

You can construct a PLINK2 case/control phenotype file where:

- `1` = control
- `2` = case

## Python

In [ ]:
analysis = pd.read_csv("1kg_annotations.txt", sep="\t")

if analysis["PurpleHair"].dtype == bool:
    purple = analysis["PurpleHair"]
else:
    purple = analysis["PurpleHair"].astype(str).str.lower().map(
        {"true": True, "false": False}
    )

purple_pheno = analysis[["#IID"]].copy()
purple_pheno["PurpleHair"] = purple.map({False: 1, True: 2})

purple_pheno.to_csv("purplehair.pheno", sep="\t", index=False)

Then run logistic/Firth GWAS:

In [ ]:
%%bash
./plink2 \
  --pfile 1kg_qc \
  --pheno purplehair.pheno \
  --pheno-name PurpleHair \
  --covar gwas.covar \
  --covar-name isFemale,PC1,PC2,PC3 \
  --glm hide-covar firth-fallback \
  --out purplehair_gwas

In [ ]:
%%bash
./plink2_mac \
  --pfile 1kg_qc \
  --pheno purplehair.pheno \
  --pheno-name PurpleHair \
  --covar gwas.covar \
  --covar-name isFemale,PC1,PC2,PC3 \
  --glm hide-covar firth-fallback \
  --out purplehair_gwas

> Now try re-running the previous plots on this newly generated summary statistics.  

---

The main conceptual lessons are:

1. **Sample IDs must be checked explicitly.**
2. **Genetic data require quality control before association testing.**
3. **PCA reveals major axes of genetic variation and ancestry.**
4. **Population stratification can generate false GWAS associations.**
5. **Principal components can help control broad-scale population structure.**
6. **A Manhattan peak identifies a region of statistical association, not necessarily a causal gene or variant.**

---

# References and further reading

- PLINK2 documentation: https://www.cog-genomics.org/plink/2.0/
- PLINK2 PCA: https://www.cog-genomics.org/plink/2.0/strat
- PLINK2 association analysis: https://www.cog-genomics.org/plink/2.0/assoc
- PLINK2 scoring: https://www.cog-genomics.org/plink/2.0/score
- 1000 Genomes Project: https://www.internationalgenome.org/